# Weather Data Processing

## Plan

1. Take three weather locations per zone, scattered across each zone 

2. Flag dates as "summer" or "winter"

3. For each station in each zone, take max or min temperature for each day depending on season

4. Pick the system-wide max or min temperature for each day. Save this value (because you'll need 2012- later) 

5. Shift +6 days and -6 days

## Execution

In [3]:
import pandas as pd
from tqdm import tqdm

In [20]:
counties = ['Albemarle_VA', 'Allen_OH', 'Baltimore_MD', 'Beaver_PA', 'Bergen_NJ', 'Blair_PA', 'Bradford_PA', 'Butler_OH', 
          'Butler_PA', 'Camden_NJ', 'Chester_PA', 'Cook_IL', 'Cuyahoga_OH', 'District_of_Columbia_DC', 'DuPage_IL', 'Erie_PA', 
          'Fairfax_VA', 'Fayette_KY', 'Franklin_OH', 'Frederick_MD', 'Hamilton_OH', 'Harford_MD', 'Hunterdon_NJ', 'Jefferson_KY',
          'Kalamazoo_MI', 'Kanawha_WV', 'Lackawanna_PA', 'Lancaster_PA', 'Lucas_OH', 'Luzerne_PA', 'Mercer_NJ', 'Middlesex_NJ', 
          'Monmouth_NJ', 'Montgomery_MD', 'Montgomery_OH', 'Montgomery_PA', 'New_Castle_DE', 'Ocean_NJ', 'Passaic_NJ',
           'Philadelphia_PA', "Prince_George's_MD", "Prince_William_VA", 'Pulaski_KY', 'Schuylkill_PA', 'Summit_OH', 
           'Virginia_Beach_VA', 'Washington_VA', 'Wicomico_MD', 'York_PA']

weather = dict()

In [21]:
for county in tqdm(counties, desc="Loading counties"):
    csv_file = f"Inputs/Weather Data/{county}.csv"
    df = pd.read_csv(csv_file)
    weather[county] = df


Loading counties: 100%|██████████| 49/49 [00:14<00:00,  3.27it/s]


In [22]:
for county, df in tqdm(weather.items(), desc="Processing counties"):
    df['date'] = pd.to_datetime(df['date'])


Processing counties: 100%|██████████| 49/49 [00:20<00:00,  2.42it/s]


In [23]:
# Create SQL Database 
conn = sqlite3.connect("weather_data_2.db")

# Loop through your dictionary of DataFrames
for county, df in tqdm(weather.items(), desc="Saving to DB"):
    # Add a 'county' column
    df['county'] = county
    
    # Append to a single table called 'hourly_weather'
    df.to_sql('hourly_weather', conn, if_exists='append', index=False)

# Commit and close
conn.commit()
conn.close()


Saving to DB:   0%|          | 0/49 [00:00<?, ?it/s]

Saving to DB: 100%|██████████| 49/49 [01:37<00:00,  1.98s/it]


In [26]:
# Sample query
# Connect to the database
conn = sqlite3.connect("weather_data_2.db")

# Query: find the maximum temperature for Baltimore
query = """
SELECT county, temperature_2m, relative_humidity_2m
FROM hourly_weather
WHERE (county, temperature_2m) IN (
    SELECT county, MAX(temperature_2m)
    FROM hourly_weather
    WHERE date BETWEEN '2012-01-01' AND '2024-12-31'
    GROUP BY county
)
GROUP BY county;"""
df = pd.read_sql(query, conn)
conn.close()

print(df)



                     county  temperature_2m  relative_humidity_2m
0              Albemarle_VA      101.942600             25.925360
1                  Allen_OH      100.318100             35.336082
2              Baltimore_MD      104.315900             23.048447
3                 Beaver_PA       99.755600             28.017060
4                 Bergen_NJ      102.613100             27.815144
5                  Blair_PA       95.055800             30.697922
6               Bradford_PA       96.547104             31.720156
7                 Butler_OH      102.389000             23.834581
8                 Butler_PA       97.876400             32.943660
9                 Camden_NJ      101.281100             27.445436
10               Chester_PA      101.007500             19.610481
11                  Cook_IL       99.229996             41.659090
12              Cuyahoga_OH       97.016890             34.187588
13  District_of_Columbia_DC      106.069110             21.498934
14        

## THI and Wind Calculation

From pages 12 and 13 of https://www.pjm.com/-/media/DotCom/documents/manuals/archive/m19/m19v35-load-forecasting-and-analysis-12-31-2021.pdf?utm_source=chatgpt.com 

In [28]:
conn = sqlite3.connect("weather_data_2.db")

cur = conn.cursor()

cur.execute("ALTER TABLE hourly_weather ADD COLUMN thi REAL;")

# Example: fill column with THI formula
cur.execute("""
UPDATE hourly_weather
SET thi = CASE
    WHEN temperature_2m >= 58 THEN
        temperature_2m - 0.55 * (1 - relative_humidity_2m/100) * (temperature_2m - 58)
    ELSE
        temperature_2m
END;
""")

# Save changes
conn.commit()


In [ ]:
# Sample query
conn = sqlite3.connect("weather_data_2.db")
query = "SELECT * FROM hourly_weather WHERE county='Baltimore_MD' AND temperature_2m > 70"
df = pd.read_sql(query, conn)
conn.close()

print(df.head())
print(type(df['date'].iloc[1]))


                        date  temperature_2m  relative_humidity_2m  \
0  1993-06-02 20:00:00+00:00        70.56590             37.372295   
1  1993-06-02 21:00:00+00:00        70.47591             36.845990   
2  1993-06-03 18:00:00+00:00        70.56590             59.554600   
3  1993-06-03 19:00:00+00:00        70.29590             55.012620   
4  1993-06-03 20:00:00+00:00        70.47591             55.950874   

   wind_speed_10m        county        thi  season  
0        7.024322  Baltimore_MD  66.237546  summer  
1        5.405955  Baltimore_MD  66.142439  summer  
2        4.529580  Baltimore_MD  67.770619  summer  
3        6.231560  Baltimore_MD  67.253518  summer  
4        6.521919  Baltimore_MD  67.453369  summer  
<class 'str'>


In [32]:
conn = sqlite3.connect("weather_data_2.db")
cur = conn.cursor()

query = """
ALTER TABLE hourly_weather ADD COLUMN season TEXT;

UPDATE hourly_weather
SET season =
    CASE
        WHEN CAST(strftime('%m', date) AS INTEGER) BETWEEN 1 AND 4
             OR CAST(strftime('%m', date) AS INTEGER) BETWEEN 10 AND 12
        THEN 'winter'
        ELSE 'summer'
    END;
"""

# Use executescript because we have multiple SQL statements
cur.executescript(query)

conn.commit()
conn.close()

In [37]:
import sqlite3

conn = sqlite3.connect("weather_data_2.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS system_hourly_max;")

conn.commit()
conn.close()


In [38]:
conn = sqlite3.connect("weather_data_2.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS system_hourly_max;")

cur.execute("""
CREATE TABLE system_hourly_max AS
SELECT 
    date,
    MAX(thi) AS system_max_thi
FROM hourly_weather
GROUP BY date
ORDER BY date;
""")

conn.commit()
conn.close()


In [40]:
# Test 

conn = sqlite3.connect("weather_data_2.db")

# Load the new system table into a DataFrame
df_test = pd.read_sql_query("""
SELECT *
FROM system_hourly_max
ORDER BY date
LIMIT 10;
""", conn)

conn.close()

# Show the first few rows
print(df_test.head())


                        date  system_max_thi
0  1993-06-01 00:00:00+00:00       70.389621
1  1993-06-01 01:00:00+00:00       69.858066
2  1993-06-01 02:00:00+00:00       69.498836
3  1993-06-01 03:00:00+00:00       69.192772
4  1993-06-01 04:00:00+00:00       69.112543


In [ ]:
conn = sqlite3.connect("weather_data_2.db")
cur = conn.cursor()

sql = """
CREATE TABLE IF NOT EXISTS system_weather_daily AS
WITH extremes AS (
    SELECT
        date,
        season,
        county,
        hourly_thi,
        CASE
            WHEN season = 'winter' THEN MIN(hourly_thi) OVER (PARTITION BY date)
            ELSE MAX(hourly_thi) OVER (PARTITION BY date)
        END AS system_thi
    FROM weather_hourly
)
SELECT date, season, county, system_thi
FROM extremes
WHERE hourly_thi = system_thi
ORDER BY day;
"""

cur.execute(sql)
conn.commit()
conn.close()


In [ ]:
# Checker 
conn = sqlite3.connect("weather_data_2.db")

# Sample the first 10 rows
query_first = "SELECT * FROM system_weather_daily ORDER BY day LIMIT 10"
df_first = pd.read_sql(query_first, conn)
print("First 10 rows:\n", df_first)

# Sample the last 10 rows
query_last = """
SELECT * FROM system_weather_daily 
ORDER BY day DESC 
LIMIT 10
"""
df_last = pd.read_sql(query_last, conn)
print("\nLast 10 rows:\n", df_last)

# Optional: check column types
print("\nColumn types:\n", df_first.dtypes)

conn.close()


First 10 rows:
           day  season             county  system_thi
0  1993-06-01  summer  Virginia_Beach_VA   70.389621
1  1993-06-02  summer         Pulaski_KY   69.587688
2  1993-06-03  summer  Virginia_Beach_VA   73.863735
3  1993-06-04  summer  Virginia_Beach_VA   74.767409
4  1993-06-05  summer  Virginia_Beach_VA   79.537216
5  1993-06-06  summer  Virginia_Beach_VA   75.532996
6  1993-06-07  summer       Jefferson_KY   79.863224
7  1993-06-08  summer  Virginia_Beach_VA   81.587835
8  1993-06-09  summer  Virginia_Beach_VA   83.975476
9  1993-06-10  summer  Virginia_Beach_VA   82.207719

Last 10 rows:
           day  season         county  system_thi
0  2025-11-15  winter  Lackawanna_PA   29.702301
1  2025-11-14  winter       Blair_PA   25.456100
2  2025-11-13  winter        Cook_IL   33.033200
3  2025-11-12  winter       Blair_PA   26.266100
4  2025-11-11  winter  Washington_VA   20.059700
5  2025-11-10  winter   Kalamazoo_MI   20.530400
6  2025-11-09  winter      DuPage_IL   28.

https://www.in.gov/iurc/files/0-2024-6-6-IURC-Meeting-Resource-Adequacy-and-Accreditation-in-PJM.pdf?utm_source=chatgpt.com

https://www.pjm.com/-/media/DotCom/documents/manuals/m20a.ashx?utm_source=chatgpt.com

32 weather years * 13 day shifts * 100 resource performance draws

In [2]:
import sqlite3
import pandas as pd

db = "weather_data_2.db"

# range(-6, 7) → -6, -5, ..., -1, 0, 1, ..., 6
shifts = list(range(-6, 7))

# But skip 0 because system_weather_daily already exists
shifts.remove(0)

conn = sqlite3.connect(db)

# Load the original "system_weather_daily"
base_df = pd.read_sql("SELECT * FROM system_hourly_max", conn)

# Ensure day is datetime
base_df['date'] = pd.to_datetime(base_df['date'])

for shift in shifts:
    # Create new shifted dataframe
    shifted_df = base_df.copy()
    shifted_df['date'] = shifted_df['date'] + pd.Timedelta(days=shift)

    # Determine table name
    if shift < 0:
        table_name = f"system_weather_hourly_minus{abs(shift)}"
    else:
        table_name = f"system_weather_hourly_plus{shift}"

    print(f"Writing table: {table_name}")

    # Write the new table to SQLite (replace if already exists)
    shifted_df.to_sql(table_name, conn, if_exists="replace", index=False)

conn.close()

print("All shifted tables created successfully!")


Writing table: system_weather_hourly_minus6
Writing table: system_weather_hourly_minus5
Writing table: system_weather_hourly_minus4
Writing table: system_weather_hourly_minus3
Writing table: system_weather_hourly_minus2
Writing table: system_weather_hourly_minus1
Writing table: system_weather_hourly_plus1
Writing table: system_weather_hourly_plus2
Writing table: system_weather_hourly_plus3
Writing table: system_weather_hourly_plus4
Writing table: system_weather_hourly_plus5
Writing table: system_weather_hourly_plus6
All shifted tables created successfully!


In [3]:
db = "weather_data_2.db"
conn = sqlite3.connect(db)
cur = conn.cursor()

sql = """
ALTER TABLE system_hourly_max RENAME TO system_weather_hourly
"""

cur.execute(sql)
conn.commit()
conn.close()



In [7]:
def preview(table):
    print(f"\n=== {table} ===")
    df = pd.read_sql(f"""
        SELECT * 
        FROM {table}
        ORDER BY day
        LIMIT 5
    """, conn)
    print(df)


In [8]:
conn = sqlite3.connect("weather_data.db")
tables = [
    "system_weather_daily_minus6",
    "system_weather_daily_minus1",
    "system_weather_daily_plus1",
    "system_weather_daily_plus6"
]

for t in tables:
    preview(t)

conn.close()



=== system_weather_daily_minus6 ===
                   day  season             county  system_thi
0  1993-05-26 00:00:00  summer  Virginia_Beach_VA   70.389621
1  1993-05-27 00:00:00  summer         Pulaski_KY   69.587688
2  1993-05-28 00:00:00  summer  Virginia_Beach_VA   73.863735
3  1993-05-29 00:00:00  summer  Virginia_Beach_VA   74.767409
4  1993-05-30 00:00:00  summer  Virginia_Beach_VA   79.537216

=== system_weather_daily_minus1 ===
                   day  season             county  system_thi
0  1993-05-31 00:00:00  summer  Virginia_Beach_VA   70.389621
1  1993-06-01 00:00:00  summer         Pulaski_KY   69.587688
2  1993-06-02 00:00:00  summer  Virginia_Beach_VA   73.863735
3  1993-06-03 00:00:00  summer  Virginia_Beach_VA   74.767409
4  1993-06-04 00:00:00  summer  Virginia_Beach_VA   79.537216

=== system_weather_daily_plus1 ===
                   day  season             county  system_thi
0  1993-06-02 00:00:00  summer  Virginia_Beach_VA   70.389621
1  1993-06-03 00:00:00

In [5]:
# Sample query
conn = sqlite3.connect("weather_data_2.db")
query = "SELECT * FROM hourly_weather WHERE county='Baltimore_MD' AND season='winter'"
df = pd.read_sql(query, conn)
conn.close()

print(df.head())
print(type(df['date'].iloc[1]))


                        date  temperature_2m  relative_humidity_2m  \
0  1993-10-01 00:00:00+00:00         51.8459             55.788820   
1  1993-10-01 01:00:00+00:00         50.1359             60.069500   
2  1993-10-01 02:00:00+00:00         48.5159             64.262955   
3  1993-10-01 03:00:00+00:00         46.6259             70.722290   
4  1993-10-01 04:00:00+00:00         45.0059             77.348620   

   wind_speed_10m        county      thi  season  
0        8.573870  Baltimore_MD  51.8459  winter  
1        7.858209  Baltimore_MD  50.1359  winter  
2        7.618947  Baltimore_MD  48.5159  winter  
3        6.487300  Baltimore_MD  46.6259  winter  
4        6.077069  Baltimore_MD  45.0059  winter  
<class 'str'>
